#### Tools: Enables agents to execute specific actions in external systems. This component provides the capability to make API calls, database updates, file operations, and other practical actions.



#### NOTE: i am using github openai for this example.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import json
import requests

In [2]:
TOKEN = os.getenv('GITHUB_TOKEN')
ENDPOINT = os.getenv('GITHUB_ENDPOINT')
MODEL = os.getenv('GITHUB_MODEL_NAME')

WEATHER_API_ENDPOINT = "https://api.open-meteo.com/v1/forecast?current=temperature_2m,wind_speed_10m"

client = OpenAI(
    base_url=ENDPOINT,
    api_key=TOKEN,
)

In [13]:
def get_weather(latitude: float, longitude: float) -> str:
    response = requests.get(f"{WEATHER_API_ENDPOINT}&latitude={latitude}&longitude={longitude}")
    data = response.json()

    return data['current']['temperature_2m']

def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    raise ValueError(f"Unknown function: {name}")

def intelligence_with_tools(prompt: str) -> str:
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get the current weather for a given location.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "latitude": {
                            "type": "number",
                            "description": "Latitude of the location."
                        },
                        "longitude": {
                            "type": "number",
                            "description": "Longitude of the location."
                        }
                    },
                    "required": ["latitude", "longitude"]
                }
            }
        }
    ]

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    # step 1: call model with tools
    response = client.chat.completions.create(
        messages=messages,
        model=MODEL,
        tools=tools,
        tool_choice="auto"
    )

    # step 2: handle function calls
    tool_calls = getattr(response.choices[0].message, 'tool_calls', [])
    
    if tool_calls:
        # Add the assistant's message with tool calls
        messages.append(response.choices[0].message)

        for tool_call in tool_calls:
            if hasattr(tool_call, 'function'):
                function_name = tool_call.function.name
                function_args = tool_call.function.arguments
                result = call_function(function_name, json.loads(function_args))

                # step 3: append function call result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(result)
                })
        
        # step 4: get final response with function results
        final_response = client.chat.completions.create(
            messages=messages,
            model=MODEL,
            tools=tools,
            tool_choice="auto"
        )

        return final_response.choices[0].message.content

    return response.choices[0].message.content

In [14]:
result = intelligence_with_tools(prompt="What's the weather like in Mumbai today?")
print("Tool Calling Output:")
print(result)

Tool Calling Output:
The current weather in Mumbai is approximately 28.2°C. If you need more details like humidity or weather conditions, let me know!
